In [ ]:
import sys
import json
import gc
import re
import random
from collections import deque
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.generation.stopping_criteria import (
    StoppingCriteria, StoppingCriteriaList,
    STOPPING_CRITERIA_INPUTS_DOCSTRING, add_start_docstrings
)
import gym
import babyai_text

ACTION_SPACE = ["turn left", "turn right", "go forward", "pick up", "drop", "toggle"]

PROMPT_HEADER_REACT = (
    "You are an expert, logical agent in a grid world. "
    "Your ONLY task is to output your reasoning and a single action in a specific format. "
    "You MUST follow this format EXACTLY. Do NOT add any extra words, explanations, or conversational text. "
    "ONLY provide the 'Thought' and 'Action' lines.\n"
    "Use this exact format:\n"
    "Thought: <your brief reasoning about the next best step>\n"
    "Action: <one action from the Valid actions list>"
)

MAX_CONTEXT_TOKENS = 4096
HISTORY_TURNS = 5
THOUGHT_TOKENS = 48
ACTION_TOKENS = 16
NUM_BEAMS = 1
DO_SAMPLE = False

USE_STUCK_INTERVENTION = True
STUCK_DETECTION_THRESHOLD = 3 

def get_valid_actions(obs_descs):
    if_obj_forward = if_wall_forward = if_door_forward = if_hold_obj = False
    for o in obs_descs:
        if 'and' not in o and o.startswith('You see') and o.endswith('1 step forward'):
            if 'wall' in o: if_wall_forward = True
            elif 'door' in o: if_door_forward = True
            else: if_obj_forward = True
        if 'carry' in o: if_hold_obj = True
    valid = ['turn left', 'turn right']
    if if_obj_forward and not if_hold_obj: valid.append('pick up')
    if not if_obj_forward and not if_wall_forward and not if_door_forward:
        valid.append('go forward')
        if if_hold_obj: valid.append('drop')
    if if_door_forward: valid.append('toggle')
    return valid if valid else ['turn left', 'turn right']

class MyStopping(StoppingCriteria):
    def __init__(self, start_length, tokenizer, stop_str):
        super().__init__()
        self.tokenizer, self.stop_str, self.start_length = tokenizer, stop_str, start_length
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs):
        decoded_text = self.tokenizer.batch_decode(input_ids[:, self.start_length:])
        return any(self.stop_str in s for s in decoded_text)

def get_stopping(start_length, stop_strs, tokenizer):
    return StoppingCriteriaList([MyStopping(start_length, tokenizer, s) for s in stop_strs])

def clamp_text_tokens(text: str, tokenizer, max_tokens: int) -> str:
    if not text or max_tokens <= 0: return text
    ids = tokenizer(text, add_special_tokens=False, return_attention_mask=False)["input_ids"]
    if len(ids) <= max_tokens: return text
    return tokenizer.decode(ids[-max_tokens:], skip_special_tokens=True).strip()

def build_react_preamble() -> str:
    return PROMPT_HEADER_REACT

def normalize_action(pred: str, valid_actions: list) -> str:
    if "action:" in pred.lower(): pred = pred.lower().split("action:")[1]
    cand = pred.strip().lower().rstrip(".")
    for head in ["action:", "action -", "action-", "action ", "action"]:
        if cand.startswith(head):
            cand = cand[len(head):].strip()
            break
    if cand in valid_actions: return cand
    for a in valid_actions:
        if a in cand: return a
    return random.choice(valid_actions)

def trim_history_lines(history_lines: list, max_turns: int) -> list:
    if not max_turns or max_turns <= 0: return []
    max_items = max_turns * 2
    return history_lines[-max_items:] if len(history_lines) > max_items else history_lines

@torch.inference_mode()
def generate_react_thought_action(model, tokenizer, messages, max_new_tokens):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompt = clamp_text_tokens(prompt, tokenizer, MAX_CONTEXT_TOKENS)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    stopping_criteria = get_stopping(inputs.input_ids.shape[-1], ['\n\n'], tokenizer)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=DO_SAMPLE,
        num_beams=NUM_BEAMS,
        stopping_criteria=stopping_criteria,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = outputs[0][inputs.input_ids.shape[-1]:]
    text = tokenizer.decode(response, skip_special_tokens=True).strip()
    return text

def react_step(model, tokenizer, system_prompt, current_observation, chat_history):
    messages = [{"role": "system", "content": system_prompt}] + chat_history
    messages.append({"role": "user", "content": current_observation})
    
    raw_output = generate_react_thought_action(model, tokenizer, messages, THOUGHT_TOKENS + ACTION_TOKENS)
    
    thought, action_text = "", ""
    lines = raw_output.split('\n')
    for line in lines:
        if line.lower().startswith("thought:"): thought = line[len("thought:"):].strip()
        elif line.lower().startswith("action:"): action_text = line[len("action:"):].strip()
    if not thought: thought = "No thought generated."
    if not action_text: action_text = raw_output
        
    return thought, action_text, {"role": "user", "content": current_observation}, {"role": "assistant", "content": raw_output}

def run_react(model, tokenizer, env, task_type, start_idx, num_task, preamble, seed):
    results_summary = {"task_type": task_type, "completed_episodes": 0, "success_rate": 0.0, "avg_reward": 0.0, "avg_steps_to_success": 0.0, "steps_to_success": [], "success_trajectories": [], "failure_trajectories": []}
    system_prompt = preamble
    
    for i in range(num_task):
        episode_num, (obs, done) = start_idx + i, (env.reset(), False)
        goal = obs[0]['mission']
        results_summary["completed_episodes"] += 1
        print(f"\n--- Starting Episode {results_summary['completed_episodes']}/{num_task} (Seed: {seed + episode_num}) ---")
        descs = obs[-1]['descriptions']
        print('Goal of the agent: ' + goal, '\nInitial Observation: ' + ', '.join(descs))
        sys.stdout.flush()

        chat_history, last_action = [], "None"
        full_trajectory_log = [f"Goal: {goal}", f"Initial Observation: {', '.join(descs)}"]
        max_steps = 30#env.max_steps
        
        stuck_detector = {}
        action_queue = deque()

        for step in range(max_steps):
            valid_actions = get_valid_actions(descs)
            obs_line = ", ".join(descs)
            action_source = "Player"
            
            if action_queue:
                action = action_queue.popleft()
                raw_action, thought = action, f"[Executing Coach's Backtracking Plan: {action}]"
                action_source = "Coach Plan"
            else:
                stuck_detector[obs_line] = stuck_detector.get(obs_line, 0) + 1
                
                stuck_warning = ""
                stuck_count = stuck_detector.get(obs_line, 0)
                if stuck_count > 1:
                    stuck_warning = f"Warning: You have visited this exact state {stuck_count} times. You might be in a loop. Try a different action than before.\n"

                if USE_STUCK_INTERVENTION and stuck_count >= STUCK_DETECTION_THRESHOLD:
                    print(f"  -> 🚨 STUCK INTERVENTION! Player visited the same state {stuck_count} times.")
                    
                    backtracking_plan = ['turn right', 'turn right']
                    if 'go forward' in valid_actions:
                        backtracking_plan.append('go forward')
                    
                    print(f"  -> 🎯 Coach Plan: Executing a backtracking plan: {backtracking_plan}")
                    action_queue.extend(backtracking_plan)
                    
                    action = action_queue.popleft()
                    raw_action, thought = action, f"[Starting Coach's Backtracking Plan: {action}]"
                    action_source = "Coach Plan"
                else:
                    current_observation_prompt = (
                        f"{stuck_warning}"
                        f"Goal: {goal}\n"
                        f"Observation: {obs_line}\n"
                        f"Valid actions: {', '.join(valid_actions)}"
                    )
                    chat_history = trim_history_lines(chat_history, HISTORY_TURNS)
                    
                    thought, raw_action, user_msg, assistant_msg = react_step(model, tokenizer, system_prompt, current_observation_prompt, chat_history)
                    chat_history.extend([user_msg, assistant_msg])

            action = normalize_action(raw_action, valid_actions)
            last_action = action
            
            a_id = ACTION_SPACE.index(action)
            _, reward, done, new_obs = env.step(a_id)
            descs = new_obs['descriptions']

            print(f"****** Step {step+1}:")
            print(f"Thought: {thought}")
            print(f"Action: {action} (Source: {action_source}, Raw: {raw_action})")
            print(f"Observation: {', '.join(descs)}")
            print("******")
            sys.stdout.flush()

            full_trajectory_log.append(f"Step {step+1}: Thought: {thought} | Action: {action} | Source: {action_source}")
            if done: break
        
        final_step = step + 1
        success = 1 if (done and reward > 0) else 0
        if success:
            results_summary["steps_to_success"].append(final_step)
        successes = len(results_summary["steps_to_success"])
        current_sr = successes / results_summary["completed_episodes"]
        print(f'Episode Result -> Success: {bool(success)}, Steps: {final_step}')
        print(f'Progress: {successes}/{results_summary["completed_episodes"]} success ({current_sr:.2%})')
        print('-----------------------------------\n')

    total_completed = results_summary["completed_episodes"]
    if total_completed > 0:
        results_summary["success_rate"] = len(results_summary["steps_to_success"]) / total_completed
    print('===== SUMMARY =====')
    print(f'Success Rate: {results_summary["success_rate"]:.2%}')
    
    return results_summary

model_name_or_path = "meta-llama/Meta-Llama-3-8B-Instruct"

task_type, num_task, start_idx = "goto", 50, 1
RESULTS_FILE_PATH = f"react_final_{model_name_or_path.split('/')[-1]}_{task_type}.json"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

torch.set_grad_enabled(False)
print(f"Loading model: {model_name_or_path} ...")
model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path, 
    dtype=torch.bfloat16,
    device_map='auto'
)
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model.eval()
print("Models loaded successfully.")

env = gym.make("BabyAI-GoToLocal-v0")
preamble = build_react_preamble()
print("Environment and preamble are ready.")

results_data = run_react(model, tokenizer, env, task_type, start_idx, num_task, preamble, SEED)

try:
    with open(RESULTS_FILE_PATH, 'w', encoding='utf-8') as f:
        def convert_numpy(obj):
            if isinstance(obj, np.number): return obj.item()
            if isinstance(obj, np.ndarray): return obj.tolist()
            return obj
        json.dump(results_data, f, indent=4, ensure_ascii=False, default=convert_numpy)
    print(f"\n✅ Results successfully saved to {RESULTS_FILE_PATH}")
except Exception as e:
    print(f"\n❌ Error saving results to file: {e}")